# Proyecto Fashion Peaks: 02 - Entrenamiento de CNN para Clasificación
Este notebook implementa el entrenamiento de la red convolucional `SpectraCNN` para estimar el área de pico mediante **clasificación discreta** (40 intervalos de 500 unidades).

### Mejoras incorporadas:
1. **Partición de datos real**: División en **Entrenamiento (70%)**, **Validación (15%)** y **Prueba ciega (15%)** con semilla reproducible.
2. **Preprocesamiento consistente**: Evita inconsistencias de transformación entre entrenamiento y evaluación.
3. **Guardado del mejor modelo (Checkpointing)**: Guarda los pesos con menor pérdida de validación para evitar sobreajuste.
4. **Evaluación rigurosa**: Cálculo de Accuracy, Matriz de Confusión y MAE (Error Absoluto Medio) en unidades reales de área sobre el conjunto de test.


In [ ]:
# 1. Configuración de entorno y conexión con Google Drive
import sys, os

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    PATH_DRIVE = '/content/gdrive/MyDrive/Fashion_Peaks_Colab'
    if os.path.exists(PATH_DRIVE):
        os.chdir(PATH_DRIVE)
    print(f"-> Conectado a Google Colab. Directorio actual: {os.getcwd()}")
else:
    print(f"-> Entorno local. Directorio actual: {os.getcwd()}")


In [ ]:
# 2. Importaciones necesarias
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
# 3. Definición del Dataset y Mapeo de Clases
def area_a_clase(area: float) -> int:
    """Convierte el valor continuo o midpoint de área a índice de clase [0, 39]"""
    return min(39, max(0, int((area - 1) // 500)))

def clase_a_area(clase: int) -> int:
    """Convierte el índice de clase [0, 39] a su centro de intervalo en unidades de área"""
    return clase * 500 + 250

class SpectraClassificationDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = self.labels_df.iloc[idx, 0]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert('L')
        area = float(self.labels_df.iloc[idx, 1])
        clase = area_a_clase(area)
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(clase, dtype=torch.long)
        return image, label


In [ ]:
# 4. Copia al disco local SSD de Colab y partición de datos
# Copiar imágenes a /content (SSD local) evita cuellos de botella de Google Drive
os.system("cp -r imagenes_entrenamiento /content/ 2>/dev/null || true")

CSV_FILE = "imagenes_entrenamiento/labels.csv"
ROOT_DIR = "/content/imagenes_entrenamiento" if os.path.exists("/content/imagenes_entrenamiento") else "imagenes_entrenamiento"

transform = transforms.Compose([
    transforms.ToTensor()
])

full_dataset = SpectraClassificationDataset(csv_file=CSV_FILE, root_dir=ROOT_DIR, transform=transform)

total_samples = len(full_dataset)
train_size = int(0.70 * total_samples)
val_size   = int(0.15 * total_samples)
test_size  = total_samples - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

print(f"Total de imágenes: {total_samples}")
print(f"  - Entrenamiento: {len(train_dataset)}")
print(f"  - Validación:    {len(val_dataset)}")
print(f"  - Prueba (Test): {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# 5. Arquitectura de la Red Convolucional (SpectraCNN)
class SpectraCNN(nn.Module):
    def __init__(self, n_classes=40):
        super(SpectraCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.AdaptiveAvgPool2d((1, 1))
        
        self.fc1 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo de entrenamiento:", device)

model = SpectraCNN(n_classes=40).to(device)


In [ ]:
# 6. Bucle de Entrenamiento y Validación
EPOCHS = 60
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

os.makedirs("modelos_guardados", exist_ok=True)
BEST_MODEL_PATH = "modelos_guardados/modelo_clasificacion_mejorado.pth"

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_loss = float('inf')

print(f"Iniciando entrenamiento por {EPOCHS} épocas...")
for epoch in range(EPOCHS):
    # --- Fase de entrenamiento ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    epoch_train_loss = running_loss / total_train
    epoch_train_acc  = correct_train / total_train

    # --- Fase de validación ---
    model.eval()
    running_val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc  = correct_val / total_val
    
    scheduler.step(epoch_val_loss)
    
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)

    # Guardar si mejora la validación
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        guardado = " [* Guardado]"
    else:
        guardado = ""

    if (epoch + 1) % 5 == 0 or epoch == 0 or guardado:
        print(f"Época [{epoch+1:02d}/{EPOCHS:02d}] "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.1f}% | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:.1f}%{guardado}")

print()
print(f"Entrenamiento finalizado. Mejor modelo guardado en: {BEST_MODEL_PATH}")


In [ ]:
# 7. Gráficas de Aprendizaje (Pérdida y Precisión)
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, val_losses, label='Val Loss')
plt.title('Pérdida por Época')
plt.xlabel('Época')
plt.ylabel('CrossEntropy Loss')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [a * 100 for a in train_accs], label='Train Acc')
plt.plot(epochs_range, [a * 100 for a in val_accs], label='Val Acc')
plt.title('Accuracy por Época')
plt.xlabel('Época')
plt.ylabel('Accuracy (%)')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# 8. Evaluación en el Conjunto de Prueba Ciego (Test Set)
# Cargar los mejores pesos guardados
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

y_true_clases, y_pred_clases = [], []
y_true_areas, y_pred_areas = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        labels = labels.cpu().numpy()
        
        y_true_clases.extend(labels)
        y_pred_clases.extend(preds)
        
        y_true_areas.extend([clase_a_area(c) for c in labels])
        y_pred_areas.extend([clase_a_area(c) for c in preds])

y_true_clases = np.array(y_true_clases)
y_pred_clases = np.array(y_pred_clases)
y_true_areas = np.array(y_true_areas)
y_pred_areas = np.array(y_pred_areas)

# Métricas finales de test
test_acc = (y_true_clases == y_pred_clases).mean() * 100
test_mae = np.mean(np.abs(y_true_areas - y_pred_areas))

print("=== RESULTADOS EN CONJUNTO DE PRUEBA (TEST SET) ===")
print(f"Accuracy Exacto de Clase : {test_acc:.2f}%")
print(f"Error Absoluto Medio (MAE): {test_mae:.2f} unidades de área")


In [ ]:
# 9. Matriz de Confusión y Análisis de Errores
# Calculamos la matriz de confusión
cm = confusion_matrix(y_true_clases, y_pred_clases, labels=list(range(40)))

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues', interpolation='nearest')
plt.title("Matriz de Confusión (Índices de Clase 0 a 39) en Test Set")
plt.colorbar()
plt.xlabel("Clase Predicha")
plt.ylabel("Clase Real")
plt.grid(False)
plt.show()

# Guardar matriz a CSV
df_cm = pd.DataFrame(cm)
df_cm.to_csv("modelos_guardados/matriz_confusion_test.csv")
print("Matriz guardada en modelos_guardados/matriz_confusion_test.csv")
